In [1]:
import ir_datasets
from collections import defaultdict
import pandas as pd
import torch
import random

MAX_QUERIES = 3000

# Load the MS MARCO passage dataset
dataset = ir_datasets.load("msmarco-passage")

# Load the queries from the development set
queries_dataset = ir_datasets.load("msmarco-passage/dev")

# Create dictionaries to store passages and queries
passages = {}
queries = {}
qrels = defaultdict(dict)

# Load passages
for passage in dataset.docs_iter():
    passages[passage.doc_id] = passage.text

# Load queries
for query in queries_dataset.queries_iter():
    queries[query.query_id] = query.text

# Load qrels
for qrel in queries_dataset.qrels_iter():
    qrels[qrel.query_id][qrel.doc_id] = qrel.relevance

# Create triplets with positives and negatives
triplets = []
passage_ids = list(passages.keys())

# Limit to max 100 queries
for query_id, doc_dict in list(qrels.items())[:MAX_QUERIES]:
    # Add positive examples
    for doc_id, relevance in doc_dict.items():
        if query_id in queries and doc_id in passages:
            triplet = (queries[query_id], passages[doc_id], relevance)
            triplets.append(triplet)
            
            # Add 8 random negative examples for this query
            negative_passages = []
            while len(negative_passages) < 8:
                neg_doc_id = random.choice(passage_ids)
                # Make sure negative example isn't actually positive
                if neg_doc_id not in doc_dict:
                    triplet = (queries[query_id], passages[neg_doc_id], 0)
                    triplets.append(triplet)
                    negative_passages.append(neg_doc_id)

# Create DataFrame from triplets
df = pd.DataFrame(triplets, columns=['query', 'passage', 'label'])

# Display first few rows
print(df.head())


                      query  \
0  . what is a corporation?   
1  . what is a corporation?   
2  . what is a corporation?   
3  . what is a corporation?   
4  . what is a corporation?   

                                             passage  label  
0  McDonald's Corporation is one of the most reco...      1  
1  Oat bran is made from the outmost bran or edib...      0  
2  Valery Rozov Climbing Cho Oyu is no small acco...      0  
3  1928: On September 11, 1928, W2XB (video) and ...      0  
4  The so called ‘cut’ of shellac is best describ...      0  


In [2]:
df["label"].value_counts()

label
0    25776
1     3222
Name: count, dtype: int64

In [3]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
model = AutoModel.from_pretrained("BAAI/bge-m3")
model.eval()



XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (position_embeddings): Embedding(8194, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementw

In [4]:
tokenized_query = tokenizer(df["query"].tolist(), padding=True, truncation=True, return_tensors="pt", max_length=512)
tokenized_passage = tokenizer(df["passage"].tolist(), padding=True, truncation=True, return_tensors="pt", max_length=512)

In [5]:
DEVICE = "cuda"
model = model.to(DEVICE)
tokenized_query = tokenized_query.to(DEVICE)
tokenized_passage = tokenized_passage.to(DEVICE)


In [6]:
from tqdm import tqdm
import time

batch_size = 128
num_samples = len(df)
embeddings_query = []
embeddings_passage = []

start_time = time.time()

with torch.no_grad():
    # Process queries in batches
    for i in tqdm(range(0, num_samples, batch_size), desc="Processing queries"):
        batch_query = {k: v[i:i+batch_size] for k,v in tokenized_query.items()}
        model_output_query = model(**batch_query)
        embeddings_query.append(model_output_query[0][:, 0])
        
    # Process passages in batches  
    for i in tqdm(range(0, num_samples, batch_size), desc="Processing passages"):
        batch_passage = {k: v[i:i+batch_size] for k,v in tokenized_passage.items()}
        model_output_passage = model(**batch_passage)
        embeddings_passage.append(model_output_passage[0][:, 0])

# Concatenate batches
embeddings_query = torch.cat(embeddings_query)
embeddings_passage = torch.cat(embeddings_passage)

dot = (embeddings_query * embeddings_passage).sum(axis=1)

df["score"] = dot.cpu().numpy()

end_time = time.time()
print(f"\nTotal processing time: {end_time - start_time:.2f} seconds")

Processing queries:   0%|                                                                                                                                                                                                                        | 0/227 [00:00<?, ?it/s]

Processing passages: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 227/227 [02:39<00:00,  1.42it/s]



Total processing time: 185.97 seconds


In [7]:
model_output_passage = model(**batch_passage)

In [8]:
batch_passage["input_ids"].shape

torch.Size([70, 363])

In [9]:
model = torch.nn.DataParallel(model)

In [10]:
preds = model(**batch_query)
preds

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.2936, -1.3278, -0.7466,  ..., -0.4554, -1.1305, -0.0209],
         [ 0.6724, -1.2898, -0.5446,  ..., -0.3842, -0.4611,  0.2817],
         [ 0.8753, -0.8517, -0.7752,  ..., -0.6780, -0.6566,  0.5353],
         ...,
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221],
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221],
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221]],

        [[ 0.2936, -1.3278, -0.7466,  ..., -0.4554, -1.1305, -0.0209],
         [ 0.6724, -1.2898, -0.5446,  ..., -0.3842, -0.4611,  0.2817],
         [ 0.8753, -0.8517, -0.7752,  ..., -0.6780, -0.6566,  0.5353],
         ...,
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221],
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221],
         [ 0.4674, -0.4852, -0.4557,  ..., -0.3745, -1.0234,  0.1221]],

        [[ 0.2936, -1.3278, -0.7466,  ..., -0.4554, -

In [12]:
from tqdm import tqdm
import time

batch_size = 128 * 8
num_samples = len(df)
embeddings_query = []
embeddings_passage = []

start_time = time.time()

with torch.no_grad():
    # Process queries in batches
    for i in tqdm(range(0, num_samples, batch_size), desc="Processing queries"):
        batch_query = {k: v[i:i+batch_size] for k,v in tokenized_query.items()}
        model_output_query = model(**batch_query)
        embeddings_query.append(model_output_query[0][:, 0])
        
    # Process passages in batches  
    for i in tqdm(range(0, num_samples, batch_size), desc="Processing passages"):
        batch_passage = {k: v[i:i+batch_size] for k,v in tokenized_passage.items()}
        model_output_passage = model(**batch_passage)
        embeddings_passage.append(model_output_passage[0][:, 0])

# Concatenate batches
embeddings_query = torch.cat(embeddings_query)
embeddings_passage = torch.cat(embeddings_passage)

dot = (embeddings_query * embeddings_passage).sum(axis=1)

df["score"] = dot.cpu().numpy()

end_time = time.time()
print(f"\nTotal processing time: {end_time - start_time:.2f} seconds")

Processing queries:   0%|                                                                                                                                                                                                                         | 0/29 [00:00<?, ?it/s]

Processing passages: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:19<00:00,  1.46it/s]



Total processing time: 27.85 seconds
